In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

batch_size = 128
epochs = 5
latent_dim = 2

(x_train, _), (x_test, _) = mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_train = x_train.reshape((len(x_train), 784))

print("Training data shape:", x_train.shape)

C:\Users\Varun\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Training data shape: (60000, 784)


In [2]:
encoder = tf.keras.Sequential([
    Dense(400, activation='relu'),
    Dense(latent_dim * 2)
])

decoder = tf.keras.Sequential([
    Dense(400, activation='relu'),
    Dense(784, activation='sigmoid')
])

In [3]:
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(batch_size)
optimizer = tf.keras.optimizers.Adam()

print("Starting VAE Training...")

for epoch in range(epochs):
    train_loss = 0
    
    for data in dataset:
        with tf.GradientTape() as tape:
            h = encoder(data)
            mu, logvar = h[:, :latent_dim], h[:, latent_dim:]
            
            z = mu + tf.exp(0.5 * logvar) * tf.random.normal(shape=tf.shape(mu))
            
            recon_batch = decoder(z)
            
            recon_loss = -tf.reduce_sum(
                data * tf.math.log(recon_batch + 1e-10) + 
                (1 - data) * tf.math.log(1 - recon_batch + 1e-10), 
                axis=-1
            )
            
            kl_loss = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=-1)
            
            loss = tf.reduce_mean(recon_loss + kl_loss)
            
        trainable_vars = encoder.trainable_variables + decoder.trainable_variables
        grads = tape.gradient(loss, trainable_vars)
        optimizer.apply_gradients(zip(grads, trainable_vars))
        
        train_loss += loss.numpy() * len(data)
        
    print(f"Epoch {epoch + 1} | Average Loss: {train_loss / len(x_train):.4f}")

Starting VAE Training...
Epoch 1 | Average Loss: 199.4148
Epoch 2 | Average Loss: 169.2189
Epoch 3 | Average Loss: 165.2653
Epoch 4 | Average Loss: 163.2494
Epoch 5 | Average Loss: 161.7658
